In [1]:
from collections import deque
import numpy as np
import torch
from actor import ActorNetwork
from critic import CriticNetwork
import numpy as np
import pystk2

class ProcessState:
    def __init__(self, max_speed=30, map_size=100, track_length=2000):
        self.frame = deque(maxlen=4)    # Window holding last 4 frames
        self.max_speed = max_speed
        self.map_size = map_size
        self.track_length = track_length

    def processObservation(self, obs):
        loc = np.array(obs["location"], dtype=np.float32) / self.map_size
        loc = np.clip(loc, -1.0, 1.0)

        vel = np.array(obs["velocity"], dtype=np.float32) / self.max_speed
        vel = np.clip(vel, -1.0, 1.0)

        front = np.array(obs["front"], dtype=np.float32) / self.map_size
        front = np.clip(front, -1.0, 1.0)

        jump = np.array([1.0 if obs["jumping"] else 0.0], dtype=np.float32)

        rotation = np.array(obs["rotation"], dtype=np.float32)

        dist = np.array([obs.get("distance_down_track", 0.0)], dtype=np.float32) / self.track_length

        state = np.concatenate([loc, vel, front, jump, rotation, dist])

        if len(self.frame) == 0:
            for _ in range(4):
                self.frame.append(state)
        else:
            self.frame.append(state)

        return np.concatenate(self.frame)

#--- STEP 1: THE PPO UPDATE FUNCTION ---

# Instantiate models with an input dimension of 60
actor_net = ActorNetwork(state_dim=60)
critic_net = CriticNetwork(state_dim=60)

actor_net.load_state_dict(torch.load('best_actor.pth'))
critic_net.load_state_dict(torch.load('best_critic.pth'))

# Set to evaluation mode for simulation
actor_net.eval()
critic_net.eval()

# --- STEP 2: THE EPISODIC WRAPPER & TRIGGER ---

for episode in range(1):
	total_episode_reward = 0.0
	
	# --- Nishant's Initialization ---
	GraphicsConfig = pystk2.GraphicsConfig.hd()
	pystk2.init(GraphicsConfig)
	WorldState = pystk2.WorldState()
	config = pystk2.RaceConfig(track='lighthouse', num_kart=1, laps=1)
	config.players[0].controller = pystk2.PlayerConfig.Controller.PLAYER_CONTROL
	race = pystk2.Race(config)
	race.start()
	
	# Track details must be loaded after race start
	track = pystk2.Track()
	track.update()
	track_length = track.length
	max_coordinate = np.max(np.abs(track.path_nodes))
	
	processor = ProcessState(max_speed=30, map_size=max_coordinate, track_length=track_length)
	RaceEnded = False
	
	# --- Nishant's Simulation Loop ---
	for step in range(1000):
		if RaceEnded:
			break
			
		WorldState.update()
		
		kart = WorldState.karts[0]
		obs = {
			"location": kart.location,
			"velocity": kart.velocity,
			"front": kart.front,
			"jumping": kart.jumping,
			"rotation": kart.rotation,
			"distance_down_track": kart.distance_down_track
		}
		
		# Mambo's Data Pipeline
		np_obs = processor.processObservation(obs=obs)
		
		# --- Phase 2 Brain Injection ---
		state_tensor = torch.FloatTensor(np_obs).unsqueeze(0)
		
		# Sanity Check for incoming engine observations
		if torch.isnan(state_tensor).any():
			print("NaN detected in engine observations! Terminating episode.")
			break
			
		with torch.no_grad():
			action_dist = actor_net(state_tensor)
			sampled_action = action_dist.mean
			state_value = critic_net(state_tensor)
			
		steer_val = torch.clamp(sampled_action[0, 0], min=-1.0, max=1.0).item()
		accel_val = torch.clamp(sampled_action[0, 1], min=0.0, max=1.0).item()
            
		print(f'SteerVal: {steer_val} |  AccelVal: {accel_val}')
		
		action = pystk2.Action()
		action.steer = steer_val
		action.acceleration = accel_val
		
		# Step the environment
		RaceEnded = not race.step(action)
		
		# Reward Calculation
		vel_x, vel_y, vel_z = obs['velocity']
		speed = (vel_x**2 + vel_y**2 + vel_z**2)**0.5
		reward = speed * 0.1
		
		if obs.get('distance_down_track', 0.0) < 0:
			reward -= 10.0
			
		total_episode_reward += reward
		
	# --- END OF EPISODE TRIGGER ---
	print(f"Episode: {episode + 1}/100 | Total Reward: {total_episode_reward:.2f}")
	
	# Critical Cleanup
	race.stop()
	del race

pystk2.clean()

..:: Antarctica Rendering Engine 2.0 ::..
SteerVal: -0.2031756341457367 |  AccelVal: 0.0
SteerVal: -0.20316839218139648 |  AccelVal: 0.0
SteerVal: -0.20297718048095703 |  AccelVal: 0.0
SteerVal: -0.2029361128807068 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0
SteerVal: -0.20286008715629578 |  AccelVal: 0.0


  wl_callback#51 still attached
  wl_surface#36 still attached
